In [4]:
import torch
import torch.nn as nn
import torch.optim as optim

### Building an AND gate


In [127]:
X = torch.tensor([[0,0],[0,1],[1,0],[1,1]]).float()
y = torch.tensor([[0],[0],[0],[1]]).float()

In [128]:
print(X)

tensor([[0., 0.],
        [0., 1.],
        [1., 0.],
        [1., 1.]])


In [129]:
model_ = nn.Sequential(
    nn.Linear(2,1),
    nn.Sigmoid()
)

In [130]:
loss_function_ = nn.BCELoss()
optimizer_ = optim.SGD(model_.parameters(), lr = 0.1)

In [131]:
for epoch in range(1000):
    optimizer_.zero_grad() # PyTorch accumulates gradients by default; this resets them to zero.
    output = model_(X) # Feeds the input throught the network.
    loss_ = loss_function_(output, y) # Calculates the loss by comparing predicted and truth values.
    loss_.backward() # Calculates gradient descents.
    optimizer_.step() # Updates the weight values.
    

In [134]:
# Evaluate Accuracy

with torch.no_grad(): # Stops gradient tracking, we don't need it for testing.
    prediction = model_(X).round() # Since the output is float value between 0 and 1, we'll round them to whole value.
    accuracy = (prediction == y).float().mean()
    print(f"Loss: {loss_:.4f} , Accurracy: {accuracy:.4f}" )
    

Loss: 0.1445 , Accurracy: 1.0000


### Building a XOR gate


In [136]:
X_ = torch.tensor([[0,0],[0,1],[1,0],[1,1]]).float()
y_ = torch.tensor([[0],[1],[1],[0]]).float()

In [137]:
# As the XOR is not linearly separable, we'll use a hidden layer to transform input space into linearly separable space
model_xor = nn.Sequential(
    nn.Linear(2,2),
    nn.Sigmoid(),
    nn.Linear(2,1),
    nn.Sigmoid(),
)

In [138]:
loss_function = nn.BCELoss()
optimizer_ = optim.Adam(model_xor.parameters(), lr = 0.1)

In [139]:
for epoch in range(10000):
    optimizer_.zero_grad()
    output = model_xor(X_)
    loss_ = loss_function(output, y_)
    loss_.backward()
    optimizer_.step()

In [142]:
with torch.no_grad():
    prediction = model_xor(X_).round() # Since the output is float value between 0 and 1, we'll round them to whole value.
    accuracy = (prediction == y_).float().mean()
    print(f"Loss: {loss_:.4f} , Accurracy: {accuracy:.4f}" )
    

Loss: 0.0000 , Accurracy: 1.0000


In [143]:
print(prediction==y_)

tensor([[True],
        [True],
        [True],
        [True]])


In [144]:
print(prediction)

tensor([[0.],
        [1.],
        [1.],
        [0.]])


## Multi Class Classification using Iris Dataset


In [145]:
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [146]:
data = load_iris()

In [147]:
# Explore the dataset
data.keys()

dict_keys(['data', 'target', 'frame', 'target_names', 'DESCR', 'feature_names', 'filename', 'data_module'])

In [148]:
data.target_names

array(['setosa', 'versicolor', 'virginica'], dtype='<U10')

In [149]:
data.data.shape

(150, 4)

In [150]:
X = data.data
y = data.target

In [151]:
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [152]:
type(X)

numpy.ndarray

In [153]:
X.shape

(150, 4)

In [154]:
"""PyTorch models work with tensors not numpy arrays"""
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.long)

In [155]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [167]:
# Our neural network will have 4 input neurons and 3 output neurons for classification
class IrisNet(nn.Module): # Base class for all PyTorch models.
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 10),
            nn.ReLU(),
            nn.Linear(10, 3),
            #No softmax here — because CrossEntropyLoss automatically applies it.
        )
        
    def forward(self, x):
        return self.net(x)
    
    

In [168]:
model = IrisNet()
loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [169]:
for epoch in range(200):
    optimizer.zero_grad()                  # Reset previous gradients
    outputs = model(X_train)               # Forward pass: predict y
    loss = loss_function(outputs, y_train)     # Compute the loss
    loss.backward()                        # Backpropagate gradients
    optimizer.step()                       # Update weights

    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

Epoch 0, Loss: 1.2740
Epoch 20, Loss: 0.6139
Epoch 40, Loss: 0.3517
Epoch 60, Loss: 0.2287
Epoch 80, Loss: 0.1486
Epoch 100, Loss: 0.1032
Epoch 120, Loss: 0.0811
Epoch 140, Loss: 0.0699
Epoch 160, Loss: 0.0634
Epoch 180, Loss: 0.0592


In [170]:
with torch.no_grad():
    test_outputs = model(X_test)
    predicted = torch.argmax(test_outputs, dim=1)
    accuracy = accuracy_score(y_test, predicted)
    print(f"\nTest Accuracy: {accuracy*100:.2f}%")


Test Accuracy: 100.00%
